In [ ]:
# =============================================================================
# Moduls
# =============================================================================
import time
import sys
import os 
import sqlite3
import subprocess
import json

import database_codes.data_sync       as data_sync
import database_codes.create_dev_data as cr_dev

from datetime        import datetime, UTC, timedelta
from IPython.display import clear_output  # Required for clearing the output in Jupyter Notebook

# =============================================================================
# Global variables for logging
# =============================================================================
logs = []

# =============================================================================
# Logging utilities
# =============================================================================

def log_message(message, section=False):
    """
    Logs a message with a timestamp and ensures the latest logs are at the top.
    Adds optional section markers for better readability.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if section:
        logs.insert(0, f"\n{'='*40}\n[{timestamp}] {message}\n{'='*40}")
    else:
        logs.insert(0, f"[{timestamp}] {message}")
    display_logs()

def display_logs():
    """
    Displays all logs with the latest entries appearing first.
    Clears the output to dynamically refresh logs.
    """
    clear_output(wait=True)  # Clear the cell output to refresh logs
    for log in logs:
        print(log)

# =============================================================================
# Sync capture and logging
# =============================================================================

def capture_and_log_sync():
    """
    Captures the output of sync_data() and logs it.
    Redirects standard output to capture print statements from sync_data().
    """
    from io import StringIO

    # Capture the standard output
    old_stdout = sys.stdout
    sys.stdout = captured_output = StringIO()
    try:
        data_sync.sync_data()
    except Exception as e:
        print(f"Error during sync: {e}")
    finally:
        sys.stdout = old_stdout

    # Log the captured output
    log_message(captured_output.getvalue(), section=True)

# =============================================================================
# Dev table maintenance logic
# =============================================================================

def ensure_dev_table_up_to_date(table_name_dev = "bchusdt_1m_dev"):
    """
    Ensures the dev table exists and is up to date.
    If the dev table does not exist, creates it.
    If the latest date in the dev table is more than one day old (based on 23:59 records),
    updates the table for each missing day at 23:59.
    """

    # -------------------------------------------------------------------------
    # DB_PATH retrieval
    # -------------------------------------------------------------------------
    repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
    DB_PATH   = json.load(open(os.path.join(repo_root, "config.json"), "r"))["db_path"]
    conn      = sqlite3.connect(DB_PATH)
    cursor    = conn.cursor()

    # -------------------------------------------------------------------------
    # Check if dev table exists
    # -------------------------------------------------------------------------
    cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name=?;", (table_name_dev,))
    exists = cursor.fetchone() is not None

    # -------------------------------------------------------------------------
    # DEV table config data
    # -------------------------------------------------------------------------
    config_path = "database_codes/config_dev_data.json"
    def dt_from_str(s): return datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=UTC)

    # Table existence check
    cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name=?;", (table_name_dev,))
    exists = cursor.fetchone() is not None

    # Target latest date: yesterday at 23:59 UTC (for up-to-date check)
    yesterday_2359 = (datetime.now(UTC) - timedelta(days=1)).replace(hour=23, minute=59, second=0, microsecond=0)

    recreate_dev_table = False

    if not exists: # if the table does not exist
        recreate_dev_table = True
        log_message(f"Dev table '{table_name_dev}' does not exist. Will be created.", section=True)
    else:
        # Table exists: check latest 23:59 record
        cursor.execute(f"SELECT MAX(open_time) FROM {table_name_dev} WHERE strftime('%H:%M', open_time) = '23:59';")
        latest_dt_str = cursor.fetchone()[0]
        if latest_dt_str is None or dt_from_str(latest_dt_str) < yesterday_2359:
            recreate_dev_table = True
            log_message(f"Dev table '{table_name_dev}' is not up to date. Will be recreated.", section=True)
        else:
            log_message(f"Dev table '{table_name_dev}' is up to date. No action required.", section=True)

    if recreate_dev_table:
        open_time_from = "2017-01-01 00:00:00"
        open_time_to   = datetime.now(UTC).replace(hour=23, minute=59, second=0, microsecond=0).strftime("%Y-%m-%d %H:%M:%S")
        cr_dev.create_dev_data_table(config_path, open_time_from, open_time_to)
        log_message(f"Dev table '{table_name_dev}' created for interval {open_time_from} to {open_time_to}.", section=True)

    conn.close()

# =============================================================================
# Scheduler logic
# =============================================================================

try:
    while True:
        log_message("Starting sync...", section=True)
        capture_and_log_sync()
        log_message("Waiting for the next cycle...")
        
        # ---------------------------------------------------------------------
        # Ensure dev table is present and up to date after each cycle
        # ---------------------------------------------------------------------
        config_path = os.path.join(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip(), "config.json")
        ensure_dev_table_up_to_date(table_name_dev="bchusdt_1m_dev")
        
        time.sleep(30)  # Wait 30 seconds
except KeyboardInterrupt:
    log_message("Scheduler interrupted. Exiting...", section=True)


[2025-10-26 08:37:01] Scheduler interrupted. Exiting...

[2025-10-26 08:36:53] Dev table 'bchusdt_1m_dev' is not up to date. Will be recreated.
[2025-10-26 08:36:53] Waiting for the next cycle...

[2025-10-26 08:36:53] Inserted rows: 2
Range inserted: 2025-10-26 09:35 -> 2025-10-26 09:36


[2025-10-26 08:36:52] Starting sync...

[2025-10-26 08:36:22] Dev table 'bchusdt_1m_dev' created for interval 2017-01-01 00:00:00 to 2025-10-26 23:59:00.

[2025-10-26 08:35:34] Dev table 'bchusdt_1m_dev' is not up to date. Will be recreated.
[2025-10-26 08:35:34] Waiting for the next cycle...

[2025-10-26 08:35:34] No new data to sync.


[2025-10-26 08:35:33] Starting sync...

[2025-10-26 08:35:03] Dev table 'bchusdt_1m_dev' created for interval 2017-01-01 00:00:00 to 2025-10-26 23:59:00.

[2025-10-26 08:34:25] Dev table 'bchusdt_1m_dev' is not up to date. Will be recreated.
[2025-10-26 08:34:24] Waiting for the next cycle...

[2025-10-26 08:34:24] Inserted rows: 4
Range inserted: 2025-10-26 09:31 -